
# FracAtlas Segmentation — U-Net++ fracture mask

Notebook này được chỉnh đúng theo yêu cầu:

- Dataset: `FracAtlas/segmentation` từ Google Drive folder đã cho.
- Task: phân đoạn vùng **fracture**.
- Model: **U-Net++**.
- Baseline: resize ảnh/mask về `256 x 256`.
- Bonus/so sánh: cấu hình giữ tỷ lệ ảnh gốc bằng padding về `256 x 256`; có thêm mode `no_resize_original` để giữ nguyên kích thước ảnh và chỉ pad cho phù hợp mạng.
- Metrics: Dice, IoU, Precision, Recall.
- Visualization: ảnh gốc, GT mask, predicted probability, predicted mask, overlay GT/Pred.
- Lưu kết quả ra CSV theo từng mode và bảng so sánh tổng hợp.

Điểm quan trọng: notebook **không suy diễn tên mask từ tên ảnh**. Mọi mask đều được đọc trực tiếp từ cột `mask_path` trong CSV.


In [ ]:

# ============================================================
# 1. MOUNT GOOGLE DRIVE / CHUẨN BỊ GDOWN
# ============================================================

from pathlib import Path
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
except Exception as e:
    print("Không mount Google Drive trong môi trường hiện tại:", e)

# gdown chỉ dùng khi không tìm thấy folder FracAtlas/segmentation trong Drive đã mount.
def ensure_gdown():
    try:
        import gdown  # noqa: F401
    except Exception:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=False)
    import gdown
    return gdown


In [ ]:

# ============================================================
# 2. IMPORT THƯ VIỆN + CẤU HÌNH CHÍNH
# ============================================================

import json
import math
import random
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# Google Drive folder từ đề bài:
FRACATLAS_DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/10cDrxrKk4nUTfB37Mvhcxn8dtascWmbZ"
FRACATLAS_DRIVE_FOLDER_ID = "10cDrxrKk4nUTfB37Mvhcxn8dtascWmbZ"

DATASET_NAME = "FracAtlas"
TASK_NAME = "segmentation_fracture"

# Các mode sẽ được train/evaluate để so sánh.
# resize_256 là baseline bắt buộc.
# keep_ratio_padding_256 giữ tỷ lệ ảnh gốc, sau đó padding về 256x256.
# no_resize_original giữ nguyên kích thước ảnh, chỉ padding lên bội số 16 để U-Net++ chạy được; mode này chậm hơn.
RUN_MODES = ["resize_256", "keep_ratio_padding_256"]
# Muốn chạy thêm đúng nghĩa không resize thì mở dòng dưới:
# RUN_MODES = ["resize_256", "keep_ratio_padding_256", "no_resize_original"]

INPUT_SIZE = 256
PAD_MULTIPLE = 16

EPOCHS = 12
BATCH_SIZE = 4
LR = 1e-3
BASE_CHANNELS = 16

# Để chạy nhanh khi demo có thể đặt MAX_TRAIN_SAMPLES / MAX_TEST_SAMPLES.
# Để nộp bài nghiêm túc nên để None.
USE_DEMO_POSITIVE_TRAIN = True
POSITIVE_TRAIN_RATIO = 0.70
MAX_TRAIN_SAMPLES = None
MAX_TEST_SAMPLES = None

PRED_THRESHOLD = 0.5
DEMO_THRESHOLD = 0.3

COPY_DATASET_TO_LOCAL = True
FORCE_RECOPY_LOCAL = False
NUM_WORKERS = 0

OUTPUT_DIR = Path("/content/fracatlas_unetpp_outputs") if os.path.exists("/content") else Path("fracatlas_unetpp_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_MODES:", RUN_MODES)
print("OUTPUT_DIR:", OUTPUT_DIR)


In [ ]:

# ============================================================
# 3. TÌM DATASET_ROOT: FracAtlas/segmentation
# ============================================================

REQUIRED_CSV = ["train.csv", "test.csv"]
OPTIONAL_CSV = ["val.csv", "labels.csv"]
REQUIRED_DIRS = ["images", "masks"]


def is_valid_segmentation_root(p):
    p = Path(p)
    if not p.exists() or not p.is_dir():
        return False
    if not all((p / name).exists() for name in REQUIRED_CSV):
        return False
    if not all((p / name).exists() for name in REQUIRED_DIRS):
        return False
    try:
        df = pd.read_csv(p / "train.csv", nrows=5)
        return "image_path" in df.columns and "mask_path" in df.columns
    except Exception:
        return False


def find_fracatlas_segmentation_root(search_roots):
    candidates = []
    for root in search_roots:
        root = Path(root)
        if not root.exists():
            continue

        direct_candidates = [
            root / "FracAtlas" / "segmentation",
            root / "fracatlas" / "segmentation",
            root / "segmentation",
        ]
        for p in direct_candidates:
            if is_valid_segmentation_root(p):
                candidates.append(p)

        # Tìm linh hoạt nhưng giới hạn vào các folder có tên segmentation để đỡ chậm.
        try:
            for p in root.rglob("segmentation"):
                if is_valid_segmentation_root(p) and "frac" in str(p).lower():
                    candidates.append(p)
        except Exception as e:
            print("Bỏ qua khi rglob trong", root, "|", e)

    # Loại trùng, ưu tiên path có FracAtlas/segmentation rõ ràng.
    unique = []
    seen = set()
    for p in candidates:
        rp = str(p.resolve()) if p.exists() else str(p)
        if rp not in seen:
            seen.add(rp)
            unique.append(p)

    if unique:
        unique = sorted(unique, key=lambda x: ("fracatlas/segmentation" not in str(x).lower().replace("\\", "/"), len(str(x))))
        return unique[0]
    return None


search_roots = [
    Path("/content/drive/MyDrive"),
    Path("/content/drive/Shareddrives"),
    Path("/content"),
    Path.cwd(),
]

DRIVE_DATA_ROOT = find_fracatlas_segmentation_root(search_roots)

if DRIVE_DATA_ROOT is None:
    print("Không tìm thấy FracAtlas/segmentation trong Drive đã mount. Thử tải bằng gdown từ folder ID...")
    gdown = ensure_gdown()
    download_root = Path("/content/FracAtlas_drive_folder") if os.path.exists("/content") else Path("FracAtlas_drive_folder")
    download_root.mkdir(parents=True, exist_ok=True)
    try:
        gdown.download_folder(
            id=FRACATLAS_DRIVE_FOLDER_ID,
            output=str(download_root),
            quiet=False,
            use_cookies=False,
            remaining_ok=True,
        )
    except TypeError:
        # Một số bản gdown cũ không có remaining_ok.
        gdown.download_folder(
            id=FRACATLAS_DRIVE_FOLDER_ID,
            output=str(download_root),
            quiet=False,
            use_cookies=False,
        )
    except Exception as e:
        print("gdown không tải được folder. Có thể folder chưa public hoặc cần add shortcut vào MyDrive.")
        print("Lỗi:", repr(e))

    DRIVE_DATA_ROOT = find_fracatlas_segmentation_root([download_root])

if DRIVE_DATA_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy dataset hợp lệ FracAtlas/segmentation. "
        "Hãy add shortcut Google Drive folder vào MyDrive hoặc đặt đúng cấu trúc: "
        "FracAtlas/segmentation/{train.csv,val.csv,test.csv,images,masks}."
    )

print("DRIVE_DATA_ROOT:", DRIVE_DATA_ROOT)
print("train.csv:", (DRIVE_DATA_ROOT / "train.csv").exists())
print("val.csv:", (DRIVE_DATA_ROOT / "val.csv").exists())
print("test.csv:", (DRIVE_DATA_ROOT / "test.csv").exists())
print("images:", (DRIVE_DATA_ROOT / "images").exists())
print("masks:", (DRIVE_DATA_ROOT / "masks").exists())


In [ ]:

# ============================================================
# 4. COPY DATASET VỀ /content THEO CSV, DÙNG ĐÚNG mask_path
# ============================================================


def safe_copy_file(src, dst, retries=3, sleep_sec=2):
    src = Path(src)
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)

    if dst.exists() and dst.stat().st_size > 0:
        return True

    last_err = None
    for attempt in range(1, retries + 1):
        try:
            shutil.copy2(src, dst)
            return True
        except Exception as e:
            last_err = e
            print(f"Copy lỗi lần {attempt}/{retries}: {src} | {e}")
            time.sleep(sleep_sec)

    print("Không copy được:", src, "->", dst, "| lỗi cuối:", last_err)
    return False


def copy_dataset_by_csv(drive_root, local_root, csv_names=("train.csv", "val.csv", "test.csv")):
    drive_root = Path(drive_root)
    local_root = Path(local_root)

    if not drive_root.exists():
        raise FileNotFoundError(f"Không thấy dataset trên Drive: {drive_root}")

    local_root.mkdir(parents=True, exist_ok=True)
    (local_root / "images").mkdir(parents=True, exist_ok=True)
    (local_root / "masks").mkdir(parents=True, exist_ok=True)

    # Copy CSV chính.
    for name in ["labels.csv", "train.csv", "val.csv", "test.csv"]:
        src_csv = drive_root / name
        if src_csv.exists():
            safe_copy_file(src_csv, local_root / name, retries=5)
        else:
            print("Cảnh báo thiếu CSV:", src_csv)

    # Đọc CSV để copy ảnh/mask đúng theo image_path và mask_path.
    rows = []
    for name in csv_names:
        csv_path = drive_root / name
        if csv_path.exists():
            df = pd.read_csv(csv_path)
            if "image_path" not in df.columns or "mask_path" not in df.columns:
                raise ValueError(f"CSV {csv_path} phải có image_path và mask_path.")
            rows.append(df)

    if len(rows) == 0:
        raise FileNotFoundError("Không đọc được train/val/test CSV từ dataset.")

    all_df = pd.concat(rows, ignore_index=True).drop_duplicates(subset=["image_path", "mask_path"])
    print("Số cặp ảnh-mask cần copy theo CSV:", len(all_df))

    copied_img = 0
    copied_mask = 0
    failed = []

    for _, row in tqdm(all_df.iterrows(), total=len(all_df), desc="Copy image/mask by CSV"):
        # QUAN TRỌNG: mask lấy trực tiếp từ row['mask_path'], không tự ghép tên file.
        src_img = drive_root / str(row["image_path"])
        src_mask = drive_root / str(row["mask_path"])
        dst_img = local_root / str(row["image_path"])
        dst_mask = local_root / str(row["mask_path"])

        ok_img = safe_copy_file(src_img, dst_img, retries=5)
        ok_mask = safe_copy_file(src_mask, dst_mask, retries=5)

        copied_img += int(ok_img)
        copied_mask += int(ok_mask)

        if not (ok_img and ok_mask):
            failed.append((str(row.get("image_id", "")), str(src_img), str(src_mask)))

    print("Copy ảnh thành công:", copied_img)
    print("Copy mask thành công:", copied_mask)
    print("Số cặp lỗi:", len(failed))
    if failed:
        print("Ví dụ lỗi:")
        for item in failed[:10]:
            print(" -", item)

    return local_root


if COPY_DATASET_TO_LOCAL:
    LOCAL_DATA_ROOT = Path("/content/FracAtlas_segmentation") if os.path.exists("/content") else Path("FracAtlas_segmentation")

    if FORCE_RECOPY_LOCAL and LOCAL_DATA_ROOT.exists():
        print("Xóa local dataset cũ:", LOCAL_DATA_ROOT)
        shutil.rmtree(LOCAL_DATA_ROOT)

    if not LOCAL_DATA_ROOT.exists() or not (LOCAL_DATA_ROOT / "train.csv").exists():
        print("Đang copy dataset từ Drive về local bằng chế độ an toàn...")
        DATA_ROOT = copy_dataset_by_csv(DRIVE_DATA_ROOT, LOCAL_DATA_ROOT)
    else:
        print("Dataset local đã tồn tại, bỏ qua copy.")
        DATA_ROOT = LOCAL_DATA_ROOT
else:
    DATA_ROOT = DRIVE_DATA_ROOT

TRAIN_CSV = DATA_ROOT / "train.csv"
VAL_CSV = DATA_ROOT / "val.csv"
TEST_CSV = DATA_ROOT / "test.csv"

print("DATA_ROOT:", DATA_ROOT)
print("train.csv:", TRAIN_CSV.exists())
print("val.csv:", VAL_CSV.exists())
print("test.csv:", TEST_CSV.exists())
print("images:", (DATA_ROOT / "images").exists())
print("masks:", (DATA_ROOT / "masks").exists())


In [ ]:

# ============================================================
# 5. ĐỌC CSV, KIỂM TRA mask_path VÀ THỐNG KÊ FRACTURE NHỎ
# ============================================================

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)
val_df = pd.read_csv(VAL_CSV) if VAL_CSV.exists() else pd.DataFrame()

print("Train/Val/Test:", len(train_df), len(val_df), len(test_df))

required_cols = ["image_path", "mask_path"]
for csv_name, df in [("train.csv", train_df), ("test.csv", test_df)]:
    missing_cols = [c for c in required_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"{csv_name} thiếu cột bắt buộc: {missing_cols}")

print("Các cột train.csv:", list(train_df.columns))
print("Các cột test.csv:", list(test_df.columns))

if "label_binary" in train_df.columns:
    print("\nPhân bố label_binary train:")
    display(train_df["label_binary"].value_counts(dropna=False))

if "label_binary" in test_df.columns:
    print("\nPhân bố label_binary test:")
    display(test_df["label_binary"].value_counts(dropna=False))


def compute_mask_area_from_csv(df, root, limit=None):
    areas = []
    iterable = df if limit is None else df.head(limit)
    for _, row in tqdm(iterable.iterrows(), total=len(iterable), desc="Compute mask_area from mask_path"):
        # QUAN TRỌNG: dùng đúng file mask_path trong CSV.
        mask_path = Path(root) / str(row["mask_path"])
        try:
            mask = Image.open(mask_path).convert("L")
            areas.append(int((np.array(mask) > 0).sum()))
        except Exception:
            areas.append(np.nan)
    return pd.Series(areas, index=iterable.index)

for name, df in [("train", train_df), ("test", test_df)]:
    if "mask_area" not in df.columns:
        print(f"CSV {name} chưa có mask_area, tính từ mask_path...")
        areas = compute_mask_area_from_csv(df, DATA_ROOT)
        if name == "train":
            train_df["mask_area"] = areas
        else:
            test_df["mask_area"] = areas

print("\nTrain có mask_area > 0:", int((train_df["mask_area"] > 0).sum()))
print("Test có mask_area > 0:", int((test_df["mask_area"] > 0).sum()))

positive_test = test_df[test_df["mask_area"] > 0].copy()
if len(positive_test) > 0:
    q25 = positive_test["mask_area"].quantile(0.25)
    q50 = positive_test["mask_area"].quantile(0.50)
    print("\nMask fracture thường nhỏ; thống kê mask_area trên test positive:")
    print("min:", int(positive_test["mask_area"].min()))
    print("q25:", float(q25))
    print("median:", float(q50))
    print("max:", int(positive_test["mask_area"].max()))


In [ ]:

# ============================================================
# 6. TẠO TRAIN CSV ƯU TIÊN ẢNH CÓ FRACTURE MASK
# ============================================================

if USE_DEMO_POSITIVE_TRAIN:
    train_df_full = train_df.copy()
    positive_df = train_df_full[train_df_full["mask_area"] > 0].copy()
    normal_df = train_df_full[train_df_full["mask_area"] == 0].copy()

    n_total = len(train_df_full) if MAX_TRAIN_SAMPLES is None else min(MAX_TRAIN_SAMPLES, len(train_df_full))
    n_positive = min(len(positive_df), max(1, int(n_total * POSITIVE_TRAIN_RATIO)))
    n_normal = max(0, n_total - n_positive)

    positive_part = positive_df.sample(n=n_positive, random_state=SEED, replace=False) if len(positive_df) > 0 else positive_df
    normal_part = normal_df.sample(n=min(n_normal, len(normal_df)), random_state=SEED, replace=False) if len(normal_df) > 0 else normal_df

    train_demo_df = pd.concat([positive_part, normal_part], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
    TRAIN_USED_CSV = OUTPUT_DIR / "train_fracatlas_positive_balanced.csv"
    train_demo_df.to_csv(TRAIN_USED_CSV, index=False, encoding="utf-8-sig")

    print("Dùng train CSV ưu tiên positive fracture:", TRAIN_USED_CSV)
    print("Số dòng:", len(train_demo_df))
    print("Positive:", int((train_demo_df["mask_area"] > 0).sum()))
    print("Normal/empty:", int((train_demo_df["mask_area"] == 0).sum()))
else:
    TRAIN_USED_CSV = TRAIN_CSV
    print("Dùng train.csv gốc:", TRAIN_USED_CSV)


In [ ]:

# ============================================================
# 7. TIỀN XỬ LÝ: BASELINE RESIZE, GIỮ TỶ LỆ, NO-RESIZE
# ============================================================


def resize_pair(img, mask, target_size=256):
    img2 = img.resize((target_size, target_size), Image.BILINEAR)
    mask2 = mask.resize((target_size, target_size), Image.NEAREST)
    meta = {
        "scale": None,
        "pad_left": 0,
        "pad_top": 0,
        "resized_width": target_size,
        "resized_height": target_size,
        "padded_width": target_size,
        "padded_height": target_size,
    }
    return img2, mask2, meta


def keep_ratio_padding_pair(img, mask, target_size=256):
    w, h = img.size
    scale = target_size / max(w, h)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))

    img_resized = img.resize((new_w, new_h), Image.BILINEAR)
    mask_resized = mask.resize((new_w, new_h), Image.NEAREST)

    img_canvas = Image.new("L", (target_size, target_size), 0)
    mask_canvas = Image.new("L", (target_size, target_size), 0)

    left = (target_size - new_w) // 2
    top = (target_size - new_h) // 2

    img_canvas.paste(img_resized, (left, top))
    mask_canvas.paste(mask_resized, (left, top))

    meta = {
        "scale": float(scale),
        "pad_left": int(left),
        "pad_top": int(top),
        "resized_width": int(new_w),
        "resized_height": int(new_h),
        "padded_width": target_size,
        "padded_height": target_size,
    }
    return img_canvas, mask_canvas, meta


def pad_to_multiple_pair(img, mask, multiple=16):
    # Không resize: giữ nguyên pixel gốc, chỉ padding phải/dưới để kích thước chia hết cho multiple.
    w, h = img.size
    new_w = int(math.ceil(w / multiple) * multiple)
    new_h = int(math.ceil(h / multiple) * multiple)

    img_canvas = Image.new("L", (new_w, new_h), 0)
    mask_canvas = Image.new("L", (new_w, new_h), 0)
    img_canvas.paste(img, (0, 0))
    mask_canvas.paste(mask, (0, 0))

    meta = {
        "scale": 1.0,
        "pad_left": 0,
        "pad_top": 0,
        "resized_width": int(w),
        "resized_height": int(h),
        "padded_width": int(new_w),
        "padded_height": int(new_h),
    }
    return img_canvas, mask_canvas, meta


def preprocess_pair(img, mask, mode="resize_256"):
    original_w, original_h = img.size

    if mode == "resize_256":
        img2, mask2, extra = resize_pair(img, mask, INPUT_SIZE)
    elif mode == "keep_ratio_padding_256":
        img2, mask2, extra = keep_ratio_padding_pair(img, mask, INPUT_SIZE)
    elif mode == "no_resize_original":
        img2, mask2, extra = pad_to_multiple_pair(img, mask, PAD_MULTIPLE)
    else:
        raise ValueError(f"Unknown preprocess mode: {mode}")

    img_np = np.array(img2).astype(np.float32) / 255.0
    mask_np = (np.array(mask2).astype(np.float32) > 0).astype(np.float32)

    x = torch.from_numpy(img_np).unsqueeze(0)
    y = torch.from_numpy(mask_np).unsqueeze(0)

    meta = {
        "original_width": int(original_w),
        "original_height": int(original_h),
        "input_width": int(img2.size[0]),
        "input_height": int(img2.size[1]),
        "preprocess_mode": mode,
    }
    meta.update(extra)
    return x, y, meta


def postprocess_pred_to_original(pred_mask_or_prob, meta, mode):
    # Đưa prediction từ không gian input trở lại kích thước ảnh gốc để overlay trực quan.
    arr = np.asarray(pred_mask_or_prob)
    original_w = int(meta["original_width"])
    original_h = int(meta["original_height"])

    if mode == "resize_256":
        pil = Image.fromarray(arr.astype(np.float32))
        pil = pil.resize((original_w, original_h), Image.BILINEAR)
        return np.array(pil)

    if mode == "keep_ratio_padding_256":
        left = int(meta["pad_left"])
        top = int(meta["pad_top"])
        rw = int(meta["resized_width"])
        rh = int(meta["resized_height"])
        cropped = arr[top:top + rh, left:left + rw]
        pil = Image.fromarray(cropped.astype(np.float32))
        pil = pil.resize((original_w, original_h), Image.BILINEAR)
        return np.array(pil)

    if mode == "no_resize_original":
        return arr[:original_h, :original_w]

    raise ValueError(mode)


In [ ]:

# ============================================================
# 8. DATASET / DATALOADER — MASK LUÔN ĐỌC TỪ mask_path
# ============================================================


def is_readable_image(path):
    try:
        with Image.open(path) as img:
            img.load()
        return True
    except Exception:
        return False


class FracAtlasSegmentationDataset(Dataset):
    def __init__(self, root, csv_path, preprocess_mode="resize_256", check_images=True):
        self.root = Path(root)
        self.csv_path = Path(csv_path)
        self.preprocess_mode = preprocess_mode
        self.df = pd.read_csv(csv_path).copy()

        if "image_path" not in self.df.columns or "mask_path" not in self.df.columns:
            raise ValueError(f"{csv_path} phải có image_path và mask_path.")

        valid_rows = []
        bad_rows = []

        for _, row in self.df.iterrows():
            img_path = self.root / str(row["image_path"])
            # QUAN TRỌNG: dùng mask_path từ CSV.
            mask_path = self.root / str(row["mask_path"])

            if not img_path.exists() or not mask_path.exists():
                bad_rows.append((row.get("image_id", ""), "missing image or mask", str(img_path), str(mask_path)))
                continue

            if check_images and (not is_readable_image(img_path) or not is_readable_image(mask_path)):
                bad_rows.append((row.get("image_id", ""), "unreadable image or mask", str(img_path), str(mask_path)))
                continue

            valid_rows.append(row)

        self.df = pd.DataFrame(valid_rows).reset_index(drop=True)

        if len(bad_rows) > 0:
            print(f"Cảnh báo: loại {len(bad_rows)} cặp ảnh-mask lỗi trong {csv_path}")
            for item in bad_rows[:10]:
                print(" -", item)

        if len(self.df) == 0:
            raise RuntimeError(f"Không có cặp ảnh-mask hợp lệ trong {csv_path}.")

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]

        img_path = self.root / str(row["image_path"])
        mask_path = self.root / str(row["mask_path"])

        with Image.open(img_path) as img:
            img = img.convert("L")
            img.load()

        with Image.open(mask_path) as mask:
            mask = mask.convert("L")
            mask.load()

        if img.size != mask.size:
            # Resize mask về ảnh gốc chỉ để đảm bảo cùng canvas; vẫn đọc đúng file mask_path.
            mask = mask.resize(img.size, Image.NEAREST)

        x, y, meta = preprocess_pair(img, mask, self.preprocess_mode)

        image_id = row.get("image_id", Path(str(row["image_path"])).stem)
        meta.update({
            "image_id": str(image_id),
            "image_path": str(row["image_path"]),
            "mask_path": str(row["mask_path"]),
            "label_binary": str(row.get("label_binary", "")),
            "label_3class": str(row.get("label_3class", "")),
            "split": str(row.get("split", "")),
        })

        if "mask_area" in row.index:
            meta["gt_mask_area_original"] = float(row["mask_area"])

        return x, y, meta


def make_loaders(preprocess_mode):
    train_ds = FracAtlasSegmentationDataset(DATA_ROOT, TRAIN_USED_CSV, preprocess_mode=preprocess_mode, check_images=True)
    test_ds = FracAtlasSegmentationDataset(DATA_ROOT, TEST_CSV, preprocess_mode=preprocess_mode, check_images=True)

    if MAX_TRAIN_SAMPLES is not None:
        train_ds = Subset(train_ds, list(range(min(MAX_TRAIN_SAMPLES, len(train_ds)))))

    if MAX_TEST_SAMPLES is not None:
        test_ds = Subset(test_ds, list(range(min(MAX_TEST_SAMPLES, len(test_ds)))))

    batch_size = BATCH_SIZE
    if preprocess_mode == "no_resize_original" and batch_size != 1:
        print("no_resize_original dùng ảnh khác kích thước, tự chuyển batch_size=1.")
        batch_size = 1

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=NUM_WORKERS)

    print(f"[{preprocess_mode}] Train/Test:", len(train_ds), len(test_ds), "| batch_size:", batch_size)
    return train_loader, test_loader, train_ds, test_ds


In [ ]:

# ============================================================
# 9. MODEL U-NET++
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNetPlusPlus(nn.Module):
    # U-Net++ 2D, grayscale input, binary output.
    def __init__(self, in_channels=1, out_channels=1, base=16):
        super().__init__()
        nb = [base, base * 2, base * 4, base * 8, base * 16]
        self.pool = nn.MaxPool2d(2, 2)

        self.conv0_0 = ConvBlock(in_channels, nb[0])
        self.conv1_0 = ConvBlock(nb[0], nb[1])
        self.conv2_0 = ConvBlock(nb[1], nb[2])
        self.conv3_0 = ConvBlock(nb[2], nb[3])
        self.conv4_0 = ConvBlock(nb[3], nb[4])

        self.conv0_1 = ConvBlock(nb[0] + nb[1], nb[0])
        self.conv1_1 = ConvBlock(nb[1] + nb[2], nb[1])
        self.conv2_1 = ConvBlock(nb[2] + nb[3], nb[2])
        self.conv3_1 = ConvBlock(nb[3] + nb[4], nb[3])

        self.conv0_2 = ConvBlock(nb[0] * 2 + nb[1], nb[0])
        self.conv1_2 = ConvBlock(nb[1] * 2 + nb[2], nb[1])
        self.conv2_2 = ConvBlock(nb[2] * 2 + nb[3], nb[2])

        self.conv0_3 = ConvBlock(nb[0] * 3 + nb[1], nb[0])
        self.conv1_3 = ConvBlock(nb[1] * 3 + nb[2], nb[1])

        self.conv0_4 = ConvBlock(nb[0] * 4 + nb[1], nb[0])
        self.final = nn.Conv2d(nb[0], out_channels, kernel_size=1)

    @staticmethod
    def up_like(x, ref):
        return F.interpolate(x, size=ref.shape[2:], mode="bilinear", align_corners=False)

    def forward(self, x):
        x0_0 = self.conv0_0(x)
        x1_0 = self.conv1_0(self.pool(x0_0))
        x2_0 = self.conv2_0(self.pool(x1_0))
        x3_0 = self.conv3_0(self.pool(x2_0))
        x4_0 = self.conv4_0(self.pool(x3_0))

        x0_1 = self.conv0_1(torch.cat([x0_0, self.up_like(x1_0, x0_0)], dim=1))
        x1_1 = self.conv1_1(torch.cat([x1_0, self.up_like(x2_0, x1_0)], dim=1))
        x2_1 = self.conv2_1(torch.cat([x2_0, self.up_like(x3_0, x2_0)], dim=1))
        x3_1 = self.conv3_1(torch.cat([x3_0, self.up_like(x4_0, x3_0)], dim=1))

        x0_2 = self.conv0_2(torch.cat([x0_0, x0_1, self.up_like(x1_1, x0_0)], dim=1))
        x1_2 = self.conv1_2(torch.cat([x1_0, x1_1, self.up_like(x2_1, x1_0)], dim=1))
        x2_2 = self.conv2_2(torch.cat([x2_0, x2_1, self.up_like(x3_1, x2_0)], dim=1))

        x0_3 = self.conv0_3(torch.cat([x0_0, x0_1, x0_2, self.up_like(x1_2, x0_0)], dim=1))
        x1_3 = self.conv1_3(torch.cat([x1_0, x1_1, x1_2, self.up_like(x2_2, x1_0)], dim=1))

        x0_4 = self.conv0_4(torch.cat([x0_0, x0_1, x0_2, x0_3, self.up_like(x1_3, x0_0)], dim=1))
        return self.final(x0_4)


def build_model():
    model = UNetPlusPlus(in_channels=1, out_channels=1, base=BASE_CHANNELS).to(DEVICE)
    return model

# Kiểm tra nhanh shape với 256x256.
_tmp_model = build_model()
with torch.no_grad():
    _tmp_out = _tmp_model(torch.zeros(1, 1, INPUT_SIZE, INPUT_SIZE, device=DEVICE))
print("Model:", _tmp_model.__class__.__name__, "| output shape:", tuple(_tmp_out.shape))
del _tmp_model, _tmp_out
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:

# ============================================================
# 10. LOSS + METRICS
# ============================================================

bce_loss = nn.BCEWithLogitsLoss()


def dice_loss_with_logits(logits, targets, smooth=1e-6):
    probs = torch.sigmoid(logits)
    targets = targets.float()
    intersection = (probs * targets).sum(dim=(1, 2, 3))
    union = probs.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))
    dice = (2 * intersection + smooth) / (union + smooth)
    return 1 - dice.mean()


def segmentation_loss(logits, targets):
    return 0.5 * bce_loss(logits, targets) + 0.5 * dice_loss_with_logits(logits, targets)


def binary_metrics(pred_mask, gt_mask, smooth=1e-7):
    pred = pred_mask.astype(bool)
    gt = gt_mask.astype(bool)

    tp = np.logical_and(pred, gt).sum()
    fp = np.logical_and(pred, ~gt).sum()
    fn = np.logical_and(~pred, gt).sum()

    pred_area = pred.sum()
    gt_area = gt.sum()

    if gt_area == 0 and pred_area == 0:
        dice = 1.0
        iou = 1.0
        precision = 1.0
        recall = 1.0
    elif gt_area == 0 and pred_area > 0:
        dice = 0.0
        iou = 0.0
        precision = 0.0
        recall = 0.0
    else:
        dice = (2 * tp + smooth) / (2 * tp + fp + fn + smooth)
        iou = (tp + smooth) / (tp + fp + fn + smooth)
        precision = (tp + smooth) / (tp + fp + smooth)
        recall = (tp + smooth) / (tp + fn + smooth)

    return {
        "dice": float(dice),
        "iou": float(iou),
        "precision": float(precision),
        "recall": float(recall),
        "gt_mask_area": float(gt_area),
        "pred_mask_area": float(pred_area),
    }


def get_meta_value(meta, key, default=""):
    value = meta.get(key, default)
    if isinstance(value, (list, tuple)):
        return value[0]
    if torch.is_tensor(value):
        value = value.detach().cpu()
        return value[0].item() if value.numel() > 0 else default
    return value


def meta_to_plain(meta):
    out = {}
    for k in meta.keys():
        v = get_meta_value(meta, k)
        if isinstance(v, np.generic):
            v = v.item()
        out[k] = v
    return out


In [ ]:

# ============================================================
# 11. TRAIN / EVALUATE MỘT EXPERIMENT
# ============================================================


def train_model(model, train_loader, epochs, lr):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        losses = []

        for x, y, meta in tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}"):
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = segmentation_loss(logits, y)
            loss.backward()
            optimizer.step()

            losses.append(float(loss.item()))

        epoch_loss = float(np.mean(losses)) if losses else float("nan")
        history.append({"epoch": epoch, "loss": epoch_loss})
        print(f"Epoch {epoch}/{epochs} | loss={epoch_loss:.4f}")

    return pd.DataFrame(history)


def evaluate_model(model, test_loader, preprocess_mode, threshold=PRED_THRESHOLD):
    model.eval()
    results = []

    with torch.no_grad():
        for x, y, meta in tqdm(test_loader, desc=f"Evaluate {preprocess_mode}"):
            x = x.to(DEVICE, non_blocking=True)
            logits = model(x)
            prob = torch.sigmoid(logits)[0, 0].cpu().numpy()

            pred_mask = (prob > threshold).astype(np.uint8)
            gt_mask = y[0, 0].numpy().astype(np.uint8)

            m = binary_metrics(pred_mask, gt_mask)
            plain_meta = meta_to_plain(meta)

            case_type = "good_case"
            if m["gt_mask_area"] > 0 and m["pred_mask_area"] == 0:
                case_type = "false_negative"
            elif m["gt_mask_area"] == 0 and m["pred_mask_area"] > 0:
                case_type = "false_positive"
            elif m["gt_mask_area"] > 0 and m["dice"] < 0.2:
                case_type = "hard_or_error_case"

            results.append({
                "group_id": "GXX",
                "student_1": "",
                "student_2": "",
                "task_id": "segmentation_fracatlas_unetpp",
                "dataset": "FracAtlas/segmentation",
                "model": "U-Net++",
                "image_id": plain_meta.get("image_id", ""),
                "image_path": plain_meta.get("image_path", ""),
                "mask_path": plain_meta.get("mask_path", ""),
                "split": plain_meta.get("split", "test"),
                "original_width": plain_meta.get("original_width", ""),
                "original_height": plain_meta.get("original_height", ""),
                "input_width": plain_meta.get("input_width", ""),
                "input_height": plain_meta.get("input_height", ""),
                "preprocess_mode": preprocess_mode,
                "is_bonus_option": "yes" if preprocess_mode != "resize_256" else "no",
                "bonus_score": 1.0 if preprocess_mode != "resize_256" else 0.0,
                "gt_mask_area": m["gt_mask_area"],
                "pred_mask_area": m["pred_mask_area"],
                "dice": m["dice"],
                "iou": m["iou"],
                "precision": m["precision"],
                "recall": m["recall"],
                "case_type": case_type,
                "note": "Fracture region can be small/low contrast; inspect overlay for hard cases.",
            })

    return pd.DataFrame(results)


def summarize_result(result_df, preprocess_mode):
    all_mean = result_df[["dice", "iou", "precision", "recall"]].mean()
    positive_df = result_df[result_df["gt_mask_area"] > 0]
    positive_mean = positive_df[["dice", "iou", "precision", "recall"]].mean() if len(positive_df) > 0 else pd.Series(dtype=float)

    row = {
        "preprocess_mode": preprocess_mode,
        "n_test": len(result_df),
        "n_positive_gt": int((result_df["gt_mask_area"] > 0).sum()),
        "dice_all": float(all_mean.get("dice", np.nan)),
        "iou_all": float(all_mean.get("iou", np.nan)),
        "precision_all": float(all_mean.get("precision", np.nan)),
        "recall_all": float(all_mean.get("recall", np.nan)),
        "dice_positive": float(positive_mean.get("dice", np.nan)) if len(positive_df) > 0 else np.nan,
        "iou_positive": float(positive_mean.get("iou", np.nan)) if len(positive_df) > 0 else np.nan,
        "precision_positive": float(positive_mean.get("precision", np.nan)) if len(positive_df) > 0 else np.nan,
        "recall_positive": float(positive_mean.get("recall", np.nan)) if len(positive_df) > 0 else np.nan,
    }
    return row


def run_experiment(preprocess_mode):
    print("\n" + "=" * 80)
    print("RUN EXPERIMENT:", preprocess_mode)
    print("=" * 80)

    mode_output_dir = OUTPUT_DIR / preprocess_mode
    mode_output_dir.mkdir(parents=True, exist_ok=True)

    train_loader, test_loader, train_ds, test_ds = make_loaders(preprocess_mode)
    model = build_model()

    history_df = train_model(model, train_loader, EPOCHS, LR)
    history_path = mode_output_dir / f"history_{preprocess_mode}.csv"
    history_df.to_csv(history_path, index=False, encoding="utf-8-sig")

    result_df = evaluate_model(model, test_loader, preprocess_mode, threshold=PRED_THRESHOLD)
    result_path = mode_output_dir / f"results_segmentation_{preprocess_mode}.csv"
    result_df.to_csv(result_path, index=False, encoding="utf-8-sig")

    model_path = mode_output_dir / f"unetpp_fracatlas_{preprocess_mode}.pth"
    torch.save(model.state_dict(), model_path)

    summary_row = summarize_result(result_df, preprocess_mode)

    print("Saved history:", history_path)
    print("Saved results:", result_path)
    print("Saved model:", model_path)
    print("Summary:")
    display(pd.DataFrame([summary_row]))

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "mode": preprocess_mode,
        "model": model,
        "history_df": history_df,
        "result_df": result_df,
        "summary": summary_row,
        "result_path": result_path,
        "model_path": model_path,
    }


In [ ]:

# ============================================================
# 12. CHẠY SO SÁNH: BASELINE RESIZE VS GIỮ TỶ LỆ / NO-RESIZE
# ============================================================

experiment_outputs = {}
summary_rows = []

for mode in RUN_MODES:
    out = run_experiment(mode)
    experiment_outputs[mode] = out
    summary_rows.append(out["summary"])

summary_df = pd.DataFrame(summary_rows)
summary_path = OUTPUT_DIR / "metrics_summary_compare_preprocess_modes.csv"
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("\nBẢNG SO SÁNH TỔNG HỢP:")
display(summary_df)
print("Saved summary:", summary_path)

# Gợi ý đọc kết quả: với fracture nhỏ, chỉ số positive subset thường quan trọng hơn all-test,
# vì all-test có thể bị kéo cao bởi nhiều mask rỗng/normal.


In [ ]:

# ============================================================
# 13. VISUALIZE: GT MASK, PRED PROB, PRED MASK, OVERLAY
# ============================================================


def load_original_pair_from_row(row):
    img_path = DATA_ROOT / str(row["image_path"])
    mask_path = DATA_ROOT / str(row["mask_path"])
    img = Image.open(img_path).convert("L")
    mask = Image.open(mask_path).convert("L")
    if img.size != mask.size:
        mask = mask.resize(img.size, Image.NEAREST)
    return img, mask


def evaluate_single_row(model, row, preprocess_mode, threshold=DEMO_THRESHOLD):
    img, mask = load_original_pair_from_row(row)
    x, y, meta = preprocess_pair(img, mask, preprocess_mode)

    model.eval()
    with torch.no_grad():
        logits = model(x.unsqueeze(0).to(DEVICE))
        prob = torch.sigmoid(logits)[0, 0].cpu().numpy()

    pred_mask = (prob > threshold).astype(np.uint8)
    gt_processed = y[0].numpy().astype(np.uint8)
    m = binary_metrics(pred_mask, gt_processed)
    return m, prob, pred_mask, gt_processed, meta


def visualize_case(model, row, preprocess_mode, threshold=DEMO_THRESHOLD):
    img, mask = load_original_pair_from_row(row)
    img_np = np.array(img)
    gt_original = (np.array(mask) > 0)

    m, pred_prob, pred_mask, gt_processed, meta = evaluate_single_row(model, row, preprocess_mode, threshold=threshold)

    pred_original_prob = postprocess_pred_to_original(pred_prob, meta, preprocess_mode)
    pred_original_mask = pred_original_prob > threshold

    image_id = row.get("image_id", Path(str(row["image_path"])).stem)
    print("preprocess_mode:", preprocess_mode)
    print("image_id:", image_id)
    print("image_path:", row["image_path"])
    print("mask_path:", row["mask_path"])
    print("GT mask area original:", int(gt_original.sum()))
    print("Metric ở threshold", threshold, ":", m)
    print("Pred prob min/max/mean:", float(pred_prob.min()), float(pred_prob.max()), float(pred_prob.mean()))

    plt.figure(figsize=(18, 4))

    plt.subplot(1, 5, 1)
    plt.imshow(img_np, cmap="gray")
    plt.title(f"Ảnh gốc\n{image_id}")
    plt.axis("off")

    plt.subplot(1, 5, 2)
    plt.imshow(gt_original, cmap="gray")
    plt.title("GT mask\nfrom mask_path")
    plt.axis("off")

    plt.subplot(1, 5, 3)
    plt.imshow(pred_prob, cmap="magma")
    plt.title("Pred probability\ninput space")
    plt.axis("off")

    plt.subplot(1, 5, 4)
    plt.imshow(pred_mask, cmap="gray")
    plt.title(f"Pred mask\nth={threshold}")
    plt.axis("off")

    plt.subplot(1, 5, 5)
    plt.imshow(img_np, cmap="gray")
    plt.imshow(gt_original, cmap="Reds", alpha=0.35)
    plt.imshow(pred_original_mask, cmap="Blues", alpha=0.25)
    plt.title("Overlay gốc\nGT đỏ, Pred xanh")
    plt.axis("off")

    plt.tight_layout()
    plt.show()


def choose_demo_rows(result_df, test_csv_df, n=3):
    rows = []
    positive = result_df[result_df["gt_mask_area"] > 0].copy()
    if len(positive) == 0:
        return test_csv_df.head(n).to_dict("records")

    # 1) Case tốt nhất theo Dice.
    best = positive.sort_values("dice", ascending=False).head(1)
    rows.extend(best.to_dict("records"))

    # 2) Fracture nhỏ/khó thấy: mask nhỏ nhất trong nhóm positive.
    small = positive.sort_values("gt_mask_area", ascending=True).head(1)
    rows.extend(small.to_dict("records"))

    # 3) False negative hoặc hard/error nếu có.
    hard = positive[positive["case_type"].isin(["false_negative", "hard_or_error_case"])]
    if len(hard) > 0:
        rows.extend(hard.sort_values("dice", ascending=True).head(1).to_dict("records"))

    # Map lại về test_csv_df để lấy đúng path/cột gốc.
    out_rows = []
    used_ids = set()
    for r in rows:
        image_id = str(r.get("image_id", ""))
        if image_id in used_ids:
            continue
        used_ids.add(image_id)
        match = test_csv_df[test_csv_df.get("image_id", "").astype(str) == image_id] if "image_id" in test_csv_df.columns else pd.DataFrame()
        if len(match) > 0:
            out_rows.append(match.iloc[0].to_dict())
        else:
            out_rows.append(r)
    return out_rows[:n]


# Visualize cho từng mode đã chạy.
test_csv_df = pd.read_csv(TEST_CSV)
for mode, out in experiment_outputs.items():
    print("\n" + "=" * 80)
    print("VISUALIZATION MODE:", mode)
    print("=" * 80)
    demo_rows = choose_demo_rows(out["result_df"], test_csv_df, n=3)
    for row in demo_rows:
        visualize_case(out["model"], row, mode, threshold=DEMO_THRESHOLD)


In [ ]:

# ============================================================
# 14. NHẬN XÉT TỰ ĐỘNG: VÙNG FRACTURE NHỎ / KHÓ THẤY
# ============================================================


def make_interpretation(summary_df, experiment_outputs):
    lines = []
    lines.append("## Nhận xét kết quả FracAtlas segmentation")
    lines.append("")
    lines.append("1. **Dice/IoU trên positive subset nên được xem kỹ hơn all-test**, vì FracAtlas có thể có nhiều ảnh normal hoặc mask rỗng; all-test dễ bị lạc quan nếu mô hình dự đoán nền tốt.")
    lines.append("2. **Vùng fracture thường nhỏ, mảnh và tương phản thấp**, nên mô hình dễ false negative hoặc dự đoán lệch vài pixel làm Dice/IoU giảm mạnh.")
    lines.append("3. **Baseline resize 256x256** nhanh và ổn định, nhưng có thể làm biến dạng tỷ lệ giải phẫu. Mode **keep_ratio_padding_256** giữ tỷ lệ tốt hơn, đổi lại có vùng padding và fracture có thể chiếm ít pixel hơn.")
    if "no_resize_original" in summary_df["preprocess_mode"].values:
        lines.append("4. **no_resize_original** giữ nhiều chi tiết nhất vì không resize, nhưng batch_size phải nhỏ, chạy chậm và tốn bộ nhớ hơn.")
    lines.append("")
    lines.append("### Bảng metric tổng hợp")
    lines.append(summary_df.to_markdown(index=False))

    # Nhận xét mode tốt nhất theo dice_positive nếu có.
    if "dice_positive" in summary_df.columns and summary_df["dice_positive"].notna().any():
        best_row = summary_df.sort_values("dice_positive", ascending=False).iloc[0]
        lines.append("")
        lines.append(f"Mode có Dice positive cao nhất trong lần chạy này: **{best_row['preprocess_mode']}** với Dice positive = **{best_row['dice_positive']:.4f}**.")
    return "\n".join(lines)

interpretation_md = make_interpretation(summary_df, experiment_outputs)
print(interpretation_md)

interpretation_path = OUTPUT_DIR / "interpretation_notes.md"
with open(interpretation_path, "w", encoding="utf-8") as f:
    f.write(interpretation_md)
print("Saved notes:", interpretation_path)


In [ ]:

# ============================================================
# 15. ĐÓNG GÓI OUTPUT
# ============================================================

zip_base = str(OUTPUT_DIR)
zip_path = shutil.make_archive(zip_base, "zip", OUTPUT_DIR)
print("Đã đóng gói output:", zip_path)
print("Các file chính:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        print("-", p)



## Checklist nộp bài

- Dataset đúng: `FracAtlas/segmentation`.
- Model đúng: `U-Net++`.
- Input baseline đúng: `256x256` với `resize_256`.
- Có cấu hình giữ tỷ lệ/không méo ảnh: `keep_ratio_padding_256`.
- Có cấu hình không resize thật sự nếu cần bật thêm: `no_resize_original`.
- Metrics có đủ: Dice, IoU, Precision, Recall.
- Có hình minh họa: GT mask, probability, predicted mask, overlay.
- CSV kết quả có cột `mask_path` và toàn bộ pipeline đọc mask từ `mask_path` trong CSV.
